In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
! pip install langchain chromadb faiss-cpu openai tiktoken langchain-openai langchain-community wikipedia

  Using cached langchain-1.3.11-py3-none-any.whl.metadata (6.0 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached langgraph-1.2.7-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.me

# 1. Based on +ve Data source

### Wikipedia Retriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever


In [2]:
retriever=WikipediaRetriever(top_k_results=2,lang="en")

In [9]:
query="Provide me information about US-IRAN war in 2026"
docs=retriever.invoke(query)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [10]:
docs

NameError: name 'docs' is not defined

In [11]:
for i,doc in enumerate(docs):
    print(f"\n---- Result {i+1} ----")
    print(f"Content:\n{doc.page_content}....")

NameError: name 'docs' is not defined

## 2. Vector Store Retriever

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [ ]:
# here our textual data
documents=[
    Document(page_content="sbi provides credit cards for their customers"),
    Document(page_content="sbi is public sector bank of india"),
    Document(page_content="sbi provides car loan at 8.50 interest rate"),
    Document(page_content="sbi also provides top up home loan for funitures and other expenses"),
    Document(page_content="SBI provides home loan at 7.50 interest rate")
]

In [ ]:
# here textual data convert into vectors
#  initialize embedding model 

vectorstore=Chroma.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings(),
    collection_name="my_collection"
)

In [ ]:
# here we compare vectors for relevant data
# Convert vectorstore into a retriever

retriever = vectorstore.as_retriever(search_kwargs={"k":1})

query="Can i get credit card from sbi"
# here we get exact result
results=retriever.invoke(query) 
print(results)

[Document(metadata={}, page_content='sbi provides credit cards for their customers')]


In [23]:
# here we get exact result

results=vectorstore.similarity_search(query,k=2)
print(results)

[Document(metadata={}, page_content='sbi provides credit cards for their customers'), Document(metadata={}, page_content='sbi provides car loan at 8.50 interest rate')]


# Based On Search Strategy

## 1. MMR (Maximum Marginal Retrievel)

In [24]:
docs=[
    Document(page_content="Langchain makes it easy to work with LLMs."),
    Document(page_content="Langchain is used to build LLM Based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="MMR helps you get diverese results when doing similarity search."),
    Document(page_content="Langchain supports Chroma, FAISS, Pinecone, and more.")
]

In [25]:
from langchain_community.vectorstores import FAISS

vectorstore_faiss=FAISS.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings()
)

In [36]:
# without mmr retirever
retriever_without_mmr=vectorstore_faiss.as_retriever(
    search_kwargs={"k":1}     # k = top results
    )


In [37]:
query="What is langchain ? "
results=retriever_without_mmr.invoke(query)
for i,doc in enumerate(results):
    print(f"\n ---- Results {i+1} ----")
    print(doc.page_content)


 ---- Results 1 ----
Langchain is used to build LLM Based applications.


In [34]:
# with mmr 

retriever_with_mmr=vectorstore_faiss.as_retriever(
    search_type="mmr",
    search_kwargs={"k":1,"lambda_mult":0.5}     # k = top results, lambda_mult = relevance-diversity balance
    )

In [35]:
query="What is langchain ? "
results=retriever_with_mmr.invoke(query)
for i,doc in enumerate(results):
    print(f"\n ---- Results {i+1} ----")
    print(doc.page_content)


 ---- Results 1 ----
Langchain is used to build LLM Based applications.


## 2. MultiQuery Retriever

In [39]:
! pip install -U langchain-classic

In [43]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

In [44]:
docs=[
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"})
]

In [46]:
vectorstore_faiss=FAISS.from_documents(documents=docs,embedding=OpenAIEmbeddings())

In [47]:
similarity_retriever=vectorstore_faiss.as_retriever(search_type="similarity",search_kwargs={"k":5})

In [48]:
llm_obj=ChatOpenAI(model="gpt-4o-mini")

In [50]:
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=similarity_retriever,
    llm=llm_obj
)

In [51]:
query="How to improve energy levels and maintain balance ? "


In [52]:
similarity_result=similarity_retriever.invoke(query)

for i, doc in enumerate(similarity_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("******************************************************")

multiquery_result=multiquery_retriever.invoke(query)

for i, doc in enumerate(multiquery_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 5 ---
Consuming leafy greens and fruits helps detox the body and improve longev

## 3. Contextual Compression Retriever

In [53]:
! pip install -U langchain-classic

In [54]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [55]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [56]:
vector=FAISS.from_documents(docs,OpenAIEmbeddings())

In [57]:
retriever=vector.as_retriever(search_kwargs={"k":2})

In [58]:
llm=ChatOpenAI(model="gpt-4o-mini")
compressor=LLMChainExtractor.from_llm(llm)

In [59]:
# now for summary

summary= ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor
)

In [60]:
query="What is photosynthesis ? "
result=summary.invoke(query)

In [61]:
for i, doc in enumerate(result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.
